# `lcz_grid_*` Tutorial — Gridded Climate & Environmental Data for LCZ Analysis

This family of functions (`lcz_grid_chirps`, `lcz_grid_era5`/
`lcz_grid_era5_global`, `lcz_grid_pdsi`, `lcz_grid_pollution_ghap`,
`lcz_grid_pollution_merra2`) downloads gridded climate/environmental
products (precipitation, reanalysis, drought indices, air quality) and
crops + reprojects them directly onto an existing **LCZ classification
GeoTIFF's grid** (from `lcz_get_map`) — exactly like `lcz_get_lst`/
`lcz_get_planetary_computer` do for satellite imagery. Output is a
per-pixel `(n_bands, H, W)` array for one city, ready for `lcz_cal_indexes`
(per-LCZ-class summary stats) or `plot_lcz_relationship` (per-pixel
distribution plots).

Every source here is free and needs **no API key** except
`lcz_grid_pollution_merra2` (NASA Earthdata Login) and `lcz_grid_era5_global`
(Copernicus CDS API key) — both are demonstrated with their auth-guard error
rather than real credentials.

Run cells top-to-bottom. §1 builds the LCZ map used by every later section.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import warnings
warnings.filterwarnings("ignore")

import numpy as np
from IPython.display import display

---
## 1 — Setup: an LCZ map

Every source here is cropped/reprojected onto a single city's LCZ
classification GeoTIFF (from `lcz_get_map`) — its footprint, CRS, and pixel
grid become the target grid.

In [ ]:
from LCZ4py import lcz_get_map

lcz_map_path = lcz_get_map(city="São Paulo, Brazil", isave_map=False)
print("LCZ map:", lcz_map_path)

---
## 2 — Precipitation: `lcz_grid_chirps`

CHIRPS v2.0 (Funk et al. 2015) — quasi-global 0.05deg (~5km) daily rainfall,
1981-present, from UCSB CHC. No auth. `resolution="monthly"` (default) sums
each month's rainfall to mm/month. The returned `LCZGridResult.array` is
`(n_bands, H, W)`, pixel-aligned to `lcz_map_path`, NaN outside the LCZ
map's valid footprint — the same convention as `lcz_get_lst`.

In [ ]:
from LCZ4py import lcz_grid_chirps

chirps_monthly = lcz_grid_chirps(lcz_map_path, resolution="monthly", years=[2023], months=[6])
print("array shape (bands, H, W):", chirps_monthly.array.shape)
print("bands:", chirps_monthly.bands)
print("variables:", chirps_monthly.variables, "| units:", chirps_monthly.units)
print("mean rainfall (mm):", np.nanmean(chirps_monthly.array))

### 2.1 — Annual resolution, for later reuse (§8)

`resolution="annual"` gives one band/year — we'll reuse this cached stack's
path directly with `plot_lcz_relationship` in §8, since it's already
aligned to the LCZ grid.

In [5]:
chirps_annual = lcz_grid_chirps(lcz_map_path, resolution="annual", years=[2023])
print(chirps_annual.path)

Using cached CHIRPS stack.
Complete: CHIRPS stack with 1 band(s), cropped to the LCZ map.
/Users/co2map/.lcz4r_cache/grid/chirps/stack/chirps_stack_03b9a10fe3bb3154.tif


---
## 3 — Reanalysis climate: `lcz_grid_era5`

ERA5-Land daily aggregates for Latin America (Saldanha, Zenodo), ~10km,
1950-2025. `vars` accepts short aliases (`t2m`, `t2m_max`, `t2m_min`, `td2m`,
`u10`, `v10`, `sp`, `tp`, or `"all"`); units are converted to
Celsius/mm/hPa automatically. One band per `{variable}_{date}` combination —
30 days x 2 vars below gives 60 bands.

In [ ]:
from LCZ4py import lcz_grid_era5

era5 = lcz_grid_era5(lcz_map_path, years=[2023], months=[6], vars=["t2m", "tp"])
print("array shape:", era5.array.shape)
print("first/last bands:", era5.bands[:2], "...", era5.bands[-2:])
t2m_bands = [i for i, b in enumerate(era5.bands) if b.startswith("t2m")]
print("mean June 2023 temperature (C):", np.nanmean(era5.array[t2m_bands]))

### 3.1 — Coverage guard: this source is Latin America only

Checked against a real downloaded file: lon -118.57..-34.07, lat
-56.75..33.35. Requesting a city outside that window raises a clear
`ValueError` pointing at `lcz_grid_era5_global` instead of failing deep
inside xarray with a confusing "no data" error.

In [6]:
berlin_map_path = lcz_get_map(city="Berlin, Germany", isave_map=False)

try:
    lcz_grid_era5(berlin_map_path, years=[2023], months=[6], vars=["t2m"])
except ValueError as e:
    print("Raised as expected:", e)

Raised as expected: This LCZ map is outside the Latin America coverage of lcz_grid_era5. Use lcz_grid_era5_global instead (requires a free Copernicus CDS API key).


### 3.2 — `lcz_grid_era5_global`: any region, via Copernicus CDS

Same variables, same `LCZGridResult` shape, but sourced from the Copernicus
Climate Data Store's global `reanalysis-era5-land-monthly-means` dataset —
covers Berlin (or anywhere else on Earth), at monthly grain only. The CDS
request itself is subset to the LCZ map's bbox (plus a small margin), so
only a small area is ever transferred, never the whole globe. Requires a
free CDS API key (https://cds.climate.copernicus.eu -> API keys), via
`cds_key`/`cds_url`, `CDSAPI_KEY`/`CDSAPI_URL` env vars, or `~/.cdsapirc` —
none configured here, so we demonstrate the auth guard instead of real data.

In [ ]:
from LCZ4py import lcz_grid_era5_global

try:
    lcz_grid_era5_global(berlin_map_path, years=[2023], months=[6], vars=["t2m"])
except ValueError as e:
    print("Raised as expected (no CDS credentials):", e)

# With credentials configured, the call is identical for any city on Earth:
#   era5_global = lcz_grid_era5_global(berlin_map_path, years=[2023], months=[6], vars=["t2m", "tp"])

---
## 4 — Meteorological drought: `lcz_grid_pdsi`

Palmer Drought Severity Index. `source="terraclimate"` (default, ~4km,
1950-2025) or `source="noaa_psl"` (2.5deg, longer record but coarse enough
that a single city's bbox may show little internal variation). Scale:
>= +4 extremely wet ... <= -4 extreme drought.

In [ ]:
from LCZ4py import lcz_grid_pdsi

pdsi = lcz_grid_pdsi(lcz_map_path, years=[2023], months=[6], source="terraclimate")
print("array shape:", pdsi.array.shape, "| bands:", pdsi.bands)
print("mean PDSI:", np.nanmean(pdsi.array))

---
## 5 — Air pollution: `lcz_grid_pollution_ghap`

GHAP (Wei et al.) AI-generated pollutant fields: PM2.5 (1km,
daily/monthly/annual, 2017-2022), O3 (10km, annual only, 2000-2020), CO
(1km, annual only, 2019-2022). No auth.

In [ ]:
from LCZ4py import lcz_grid_pollution_ghap

ghap = lcz_grid_pollution_ghap(lcz_map_path, pollutants=["pm25"], resolution="annual", years=[2020])
print("array shape:", ghap.array.shape, "| bands:", ghap.bands)
print("mean PM2.5 (ug/m3):", np.nanmean(ghap.array))

### 5.1 — Map plot

`ghap.path` is already a GeoTIFF aligned to the LCZ grid, so it can be
handed straight to `lcz_plot_map` (the same continuous-raster path
`lcz_get_lst` stacks take) — no separate save step needed. `data_type="auto"`
detects the float PM2.5 values and switches from the LCZ categorical
colormap to a continuous one.

In [ ]:
from LCZ4py import lcz_plot_map

ghap_map = lcz_plot_map(ghap.path, band=ghap.bands[0], title="GHAP PM2.5, 2020 (ug/m3)", colorscale="Reds")
display(ghap_map.fig)

### 5.2 — Relationship with LCZ classes: `lcz_cal_indexes`

Per-class summary (n_pixels/mean/std/...) — built-up classes (compact/open
low- and mid-rise) tend to show higher PM2.5 than natural-land classes,
consistent with traffic/combustion sources concentrated in the urban core.

In [ ]:
from LCZ4py import lcz_cal_indexes

ghap_by_lcz = lcz_cal_indexes(lcz_map_path, ghap, variable_name="pm25_ug_m3")
print(ghap_by_lcz.df.select(["lcz", "lcz_name", "n_pixels", "mean", "std"]))
display(ghap_by_lcz.fig)

### 5.3 — Relationship with LCZ classes: `plot_lcz_relationship`

The per-pixel counterpart — every valid pixel's PM2.5 value, grouped by LCZ
class, as a violin plot (distribution shape, not just mean/std).

In [ ]:
from LCZ4py import plot_lcz_relationship

fig_ghap_violin = plot_lcz_relationship(
    lcz_map_path, ghap.path, variable_name="pm25_ug_m3", plot_type="heatmap",
)
display(fig_ghap_violin)

---
## 6 — Air pollution: `lcz_grid_pollution_merra2`

MERRA-2 aerosol reanalysis (Gelaro et al. 2017) — requires a free **NASA
Earthdata Login** (https://urs.earthdata.nasa.gov), via
`earthdata_user`/`earthdata_pass`, the `EARTHDATA_USER`/`EARTHDATA_PASSWORD`
env vars, or a `.netrc` path. No credentials are configured in this
environment, so we demonstrate the auth guard instead of real data.

In [ ]:
from LCZ4py import lcz_grid_pollution_merra2

try:
    lcz_grid_pollution_merra2(lcz_map_path, pollutants=["pm25"], years=[2023], months=[6])
except ValueError as e:
    print("Raised as expected (no Earthdata credentials):", e)

---
## 7 — `lcz_cal_indexes`: summarize a gridded variable by LCZ class

The aggregation counterpart to `lcz_cal_area` (which summarizes *area* per
LCZ class): here we summarize a co-registered climate variable instead —
one row of `n_pixels, mean, std, min, max, median` per LCZ class present in
the city.

In [ ]:
from LCZ4py import lcz_cal_indexes

rain_by_lcz = lcz_cal_indexes(lcz_map_path, chirps_annual, variable_name="rainfall_2023_mm")
print(rain_by_lcz.df.select(["lcz", "lcz_name", "n_pixels", "mean", "std"]))
display(rain_by_lcz.fig)

### 7.1 — Picking a single band from a multi-band stack

`band` accepts an index or one of `grid_result.bands`'s labels; omit it to
pool every band together (as above, `chirps_annual` only has one band).

In [15]:
t2m_by_lcz = lcz_cal_indexes(lcz_map_path, era5, band="t2m_2023-06-15", variable_name="t2m_2023-06-15_C")
print(t2m_by_lcz.df.select(["lcz", "lcz_name", "mean", "std"]))

shape: (14, 4)
┌─────┬────────────────────┬───────────┬──────────┐
│ lcz ┆ lcz_name           ┆ mean      ┆ std      │
│ --- ┆ ---                ┆ ---       ┆ ---      │
│ i16 ┆ str                ┆ f64       ┆ f64      │
╞═════╪════════════════════╪═══════════╪══════════╡
│ 1   ┆ Compact highrise   ┆ 15.846639 ┆ 0.026572 │
│ 2   ┆ Compact midrise    ┆ 15.848192 ┆ 0.069715 │
│ 3   ┆ Compact lowrise    ┆ 15.8489   ┆ 0.176122 │
│ 4   ┆ Open highrise      ┆ 15.800813 ┆ 0.101398 │
│ 5   ┆ Open midrise       ┆ 15.824528 ┆ 0.161531 │
│ …   ┆ …                  ┆ …         ┆ …        │
│ 12  ┆ Scattered trees    ┆ 15.854641 ┆ 0.369904 │
│ 14  ┆ Low plants         ┆ 15.714973 ┆ 0.16696  │
│ 15  ┆ Bare rock or paved ┆ 15.783931 ┆ 0.505765 │
│ 16  ┆ Bare soil or sand  ┆ 15.583499 ┆ 0.137969 │
│ 17  ┆ Water              ┆ 15.749218 ┆ 0.185385 │
└─────┴────────────────────┴───────────┴──────────┘


---
## 8 — `plot_lcz_relationship`: pixel-level distribution

Same idea as `lcz_cal_indexes`, but plots every valid pixel's value instead
of just the summary — box/violin/scatter/heatmap. Since `chirps_annual`'s
cache file is *already* aligned to `lcz_map_path`'s exact grid, this call
skips the internal reproject step entirely (same CRS/transform/shape check
in `plot_lcz_relationship._align_to_lcz`).

In [ ]:
from LCZ4py import plot_lcz_relationship

fig_box = plot_lcz_relationship(
    lcz_map_path, chirps_annual.path, variable_name="rainfall_2023_mm", plot_type="violin",
)
display(fig_box)

In [ ]:
fig_heatmap = plot_lcz_relationship(
    lcz_map_path, chirps_annual.path, variable_name="rainfall_2023_mm", plot_type="violin",
)
display(fig_heatmap)

---
## 9 — Cache behavior

Every raster `lcz_grid_*` function caches both the raw downloaded file
(raster/NetCDF) and the LCZ-grid-aligned stack (keyed by a hash of the
request params + the LCZ map's bbox). Repeating an identical call skips
both the network round-trip and the crop/reproject step.

In [18]:
import time

t0 = time.time()
_ = lcz_grid_chirps(lcz_map_path, resolution="monthly", years=[2023], months=[6])
print(f"cached call: {time.time() - t0:.3f}s")

Using cached CHIRPS stack.
Complete: CHIRPS stack with 1 band(s), cropped to the LCZ map.
cached call: 0.012s


---
## 10 — Error handling

Invalid parameters raise `ValueError` with a clear message rather than
failing deep inside a download or a reproject.

In [19]:
try:
    lcz_grid_chirps(lcz_map_path, resolution="hourly")
except ValueError as e:
    print("Raised as expected:", e)

try:
    lcz_grid_chirps(lcz_map_path, resampling="lanczos")
except ValueError as e:
    print("Raised as expected:", e)

Raised as expected: Invalid resolution: hourly. Use: monthly, daily, annual.
Raised as expected: Invalid resampling method. Use: average, bilinear, nearest.


---
## 11 — End-to-end: a per-city LCZ climate profile

Chain several raster sources plus `lcz_cal_indexes` into one combined
per-LCZ-class summary table for the same city — the shape a per-city urban
climate report would want.

In [20]:
profile_frames = {
    "rainfall_2023_mm": lcz_cal_indexes(lcz_map_path, chirps_annual, iplot=False)[["lcz", "lcz_name", "mean"]],
    "pdsi": lcz_cal_indexes(lcz_map_path, pdsi, iplot=False)[["lcz", "mean"]],
    "pm25_ug_m3": lcz_cal_indexes(lcz_map_path, ghap, iplot=False)[["lcz", "mean"]],
}

profile = profile_frames["rainfall_2023_mm"].rename({"mean": "rainfall_2023_mm"})
profile = profile.join(profile_frames["pdsi"].rename({"mean": "pdsi"}), on="lcz")
profile = profile.join(profile_frames["pm25_ug_m3"].rename({"mean": "pm25_ug_m3"}), on="lcz")
profile

lcz,lcz_name,rainfall_2023_mm,pdsi,pm25_ug_m3
i16,str,f64,f64,f64
1,"""Compact highrise""",1347.811198,-1.763313,19.111467
2,"""Compact midrise""",1318.200837,-1.780694,19.052829
3,"""Compact lowrise""",1397.949588,-1.59946,18.704979
4,"""Open highrise""",1341.057036,-1.721019,19.180984
5,"""Open midrise""",1366.542773,-1.635891,18.929532
…,…,…,…,…
12,"""Scattered trees""",1456.768318,-1.371769,16.56867
14,"""Low plants""",1401.695636,-1.483884,18.398705
15,"""Bare rock or paved""",1394.574337,-1.798696,16.375631


---
## Summary

- **`lcz_grid_chirps`** `(x, resolution, years, months, resampling, ...)`
  -> `LCZGridResult` with `array` `(n_bands, H, W)` mm, `bands` = ISO dates.
  CHIRPS v2.0, 0.05deg, 1981-present.
- **`lcz_grid_era5`** `(x, years, months, vars, resampling, ...)`
  -> `array` `(n_bands, H, W)`, `bands` = `"{var}_{date}"`. ERA5-Land, ~10km,
  daily, 1950-2025, **Latin America only** (raises `ValueError` outside that
  bbox); no auth.
- **`lcz_grid_era5_global`** `(x, years, months, vars, resampling, cds_key, cds_url, ...)`
  -> same shape, monthly grain only, **any region on Earth**, via Copernicus
  CDS (`reanalysis-era5-land-monthly-means`). Requires a free CDS API key.
- **`lcz_grid_pdsi`** `(x, years, months, source, resampling, ...)`
  -> `array` `(n_months, H, W)`, `bands` = ISO dates. `source="terraclimate"`
  (~4km) or `"noaa_psl"` (2.5deg).
- **`lcz_grid_pollution_ghap`** `(x, pollutants, resolution, years, months, resampling, ...)`
  -> `array` `(n_bands, H, W)`, `bands` = `"{pollutant}_{date}"`.
- **`lcz_grid_pollution_merra2`** `(x, pollutants, years, months, resampling, earthdata_user/pass, netrc_path, ...)`
  -> `array` `(n_bands, H, W)`, `bands` = `"{pollutant}_{date}"`. Requires
  NASA Earthdata Login.
- **`lcz_cal_indexes`** `(x, grid_result, band, variable_name, ...)`
  -> `LCZIndexesResult(df, fig)`: n_pixels/mean/std/min/max/median per LCZ class.
- **`plot_lcz_relationship`** `(lcz_path, variable_path, variable_name, plot_type, ...)`
  -> matplotlib `Figure` by default (`renderer="plotly"` for the interactive
  Plotly figure), per-pixel distribution by LCZ class
  (box/violin/scatter/heatmap).

All functions accept `lang="en"/"pt"/"es"/"zh"` for message translation and
cache raw downloads under `~/.lcz4r_cache/grid/<source>/` and the
LCZ-grid-aligned stack under `<source>/stack/`.